# Load and plot DPR-DSA-03

Example walkthrough for the convening sample: find a **DPR-DSA-03** continuum
radio camera mosaic by product name under ``data/``, load the FITS image, and plot it.

`DPR-DSA-03` is a calibrated continuum mosaic (FITS, Jy/beam). **Download the sample into `data/`** (see `data/README.md`). Filenames should include the product
id, e.g. `mock_DPR-DSA-03.fits`.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.visualization import simple_norm
from astropy.wcs import WCS

from convening2026 import data_dir, find_product

# Product id used to locate the sample file by name under data/
PRODUCT = "DPR-DSA-03"

data_dir()


In [ ]:
fits_path = find_product(PRODUCT, pattern=f"**/*{PRODUCT}*.fits")
print(f"Using: {fits_path}")


In [ ]:
with fits.open(fits_path) as hdul:
    hdul.info()
    primary = hdul[0]
    header = primary.header
    image = np.asarray(primary.data, dtype=float)

print(f"shape: {image.shape}")
print(f"BUNIT: {header.get('BUNIT', 'n/a')}")
print(f"MOSAICID: {header.get('MOSAICID', 'n/a')}")
print(f"OBJECT: {header.get('OBJECT', 'n/a')}")


In [ ]:
# Prefer a WCS axes when the header provides a celestial projection
wcs = None
try:
    candidate = WCS(header)
    if candidate.naxis >= 2 and candidate.has_celestial:
        wcs = candidate.celestial
except Exception as exc:
    print(f"WCS not available ({exc}); plotting in pixel coordinates.")

fig = plt.figure(figsize=(8, 7))
ax = fig.add_subplot(1, 1, 1, projection=wcs) if wcs is not None else fig.add_subplot(1, 1, 1)

finite = image[np.isfinite(image)]
norm = simple_norm(finite if finite.size else image, "asinh", percent=99.5)
im = ax.imshow(image, origin="lower", cmap="gray", norm=norm)

title = f"{PRODUCT}: {fits_path.name}"
if header.get("MOSAICID"):
    title += f"\n{header['MOSAICID']}"
ax.set_title(title)

if wcs is not None:
    ax.set_xlabel("RA")
    ax.set_ylabel("Dec")
    ax.coords.grid(True, color="white", alpha=0.3, linestyle=":")
else:
    ax.set_xlabel("x (pixel)")
    ax.set_ylabel("y (pixel)")

cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label(header.get("BUNIT", "pixel value"))
plt.tight_layout()
plt.show()
